### Connect the Google Drive to the notebook

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.chdir("/content/drive/MyDrive/dataset")

### Image path load

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import pandas as pd
import cv2

In [ ]:
annotation_file = 'data/Selfie-dataset/selfie_dataset.txt'

# Initialize lists to hold data
selfie_img_path = []
popularity_scores = []
attributes_list = []

# Open the file and read line by line
with open(annotation_file, 'r') as file:
    for line in file:
        # Split the line into parts
        parts = line.strip().split()

        # Extract the image name and popularity score
        current_selfie_img_path = 'data/Selfie-dataset/images/' + parts[0] + '.jpg'
        popularity_score = float(parts[1])

        # Convert attribute values from strings to integers
        attributes = [int(attr) if int(attr) == 1 else 0 for attr in parts[2:]]

        # Append data to the lists
        selfie_img_path.append(current_selfie_img_path)
        popularity_scores.append(popularity_score)
        attributes_list.append(attributes)


### Import

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

### Preparing Data

In [ ]:
# Split the data into training and testing sets
train_img_paths, test_img_paths, train_labels, test_labels = train_test_split(
    selfie_img_path,
    attributes_list,
    test_size=0.2,
    random_state=42
)

def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [128, 128])
    image /= 255.0  # normalize to [0,1] range
    return image, label

# Create a dataset from tensor slices
train_dataset = tf.data.Dataset.from_tensor_slices((train_img_paths, train_labels))
test_dataset = tf.data.Dataset.from_tensor_slices((test_img_paths, test_labels))

# Map the load and preprocess function
train_dataset = train_dataset.map(load_and_preprocess_image)
test_dataset = test_dataset.map(load_and_preprocess_image)

### CNN Model

#### Hyperparameters

In [ ]:
num_attributes = 36
num_epochs = 10
batch_size = 32
learning_rate = 0.01

In [ ]:
# Batch and shuffle dataset
train_dataset = train_dataset.shuffle(buffer_size=len(train_img_paths)).batch(batch_size)
test_dataset = test_dataset.batch(batch_size)

#### Define CNN Model 

In [ ]:
# Build the CNN model for multi-label classification
image_size = (128, 128)
selfie_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(image_size[0], image_size[1], 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(num_attributes, activation='sigmoid')  # Sigmoid activation for multi-label classification
])

# Compile the model
selfie_model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])

#### Train & Test

In [ ]:
# Train the model
selfie_model.fit(
    train_dataset,
    epochs=num_epochs
    # validation_data=test_dataset
)

In [ ]:
# Evaluate the model on the test dataset
test_loss, test_accuracy = selfie_model.evaluate(test_dataset, verbose=0)

print(f"Test loss: {test_loss}")
print(f"Test accuracy: {test_accuracy}")

# # If you want to make predictions
# predictions = selfie_model.predict(test_dataset)